In [1]:
import os

os.environ["OMP_NUM_THREADS"] = "6"
os.environ["MKL_NUM_THREADS"] = "6"
os.environ["OPENBLAS_NUM_THREADS"] = "6"
os.environ["NUMEXPR_NUM_THREADS"] = "6"

import torch
torch.set_num_threads(6)
torch.set_num_interop_threads(1)

In [2]:
import sys

import torch

import pandas as pd

import pm4py

from config.feature_config import FeatureConfig
from config.ga_config import GAConfig

from utils.general_utils import set_stdout_to_file, set_seed
from utils.feature_utils import df_to_sequence_array

from model.preprocessor import PreprocessorArtifacts

from model.next_event_model import ProcessLSTM
from model.model_wrapper import ModelWrapper

from ga_search.search import CounterfactualGA

from process.engine import ProcessModelConstraintEngine
from process.experimenter import ExperimentHandler

### --- Load Dataset & Models ---

In [3]:
set_seed(seed=777)

In [4]:
df = pd.read_excel(
    "../../../data/road_fine.xlsx",
    engine="openpyxl",
    keep_default_na=False,
    dtype={
        "case:concept:name": "string",
        "concept:name": "string",
        "lifecycle:transition": "string",
        "org:resource": "string",
        "dismissal": "string",
        "vehicleClass": "string",
        "notificationType": "string",
        "lastSent": "string",
        "amount": "float32",
        "totalPaymentAmount": "float32",
        "article": "float32",
        "points": "float32",
        "expense": "float32",
        "paymentAmount": "float32",
        "time_delta": "float32",
    }
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [5]:
df.head(20)

,case:concept:name,time:timestamp,amount,article,concept:name,dismissal,expense,lastSent,lifecycle:transition,notificationType,org:resource,paymentAmount,points,time_delta,totalPaymentAmount,vehicleClass
0,A1,2006-07-24,35.0,157.0,Create Fine,NIL,0.0,NA,complete,NA,561,0.0,0.0,0.0,0.0,A
1,A1,2006-12-05,0.0,0.0,Send Fine,NA,11.0,NA,complete,NA,NA,0.0,0.0,11577600.0,0.0,NA
2,A100,2006-08-02,35.0,157.0,Create Fine,NIL,0.0,NA,complete,NA,561,0.0,0.0,0.0,0.0,A
3,A100,2006-12-12,0.0,0.0,Send Fine,NA,11.0,NA,complete,NA,NA,0.0,0.0,11404800.0,0.0,NA
4,A100,2007-01-15,0.0,0.0,Insert Fine Notification,NA,0.0,P,complete,P,NA,0.0,0.0,2937600.0,0.0,NA
5,A100,2007-03-16,71.5,0.0,Add penalty,NA,0.0,NA,complete,NA,NA,0.0,0.0,5184000.0,0.0,NA
6,A100,2009-03-30,0.0,0.0,Send for Credit Collection,NA,0.0,NA,complete,NA,NA,0.0,0.0,64368000.0,0.0,NA
7,A10000,2007-03-09,36.0,157.0,Create Fine,NIL,0.0,NA,complete,NA,561,0.0,0.0,0.0,0.0,A
8,A10000,2007-07-17,0.0,0.0,Send Fine,NA,13.0,NA,complete,NA,NA,0.0,0.0,11232000.0,0.0,NA
9,A10000,2007-08-02,0.0,0.0,Insert Fine Notification,NA,0.0,P,complete,P,NA,0.0,0.0,1382400.0,0.0,NA


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [7]:
feature_config = FeatureConfig.load(
    path = "../pretrained_models/"
)

In [8]:
feature_config.summary()

  activity_feature:    concept:name
  feature_order:       ['amount', 'article', 'concept:name', 'dismissal', 'expense', 'lastSent', 'lifecycle:transition', 'notificationType', 'org:resource', 'paymentAmount', 'points', 'time_delta', 'totalPaymentAmount', 'vehicleClass']
--------------------------------------------------------------------------------------------------------------------------------------
Feature                        Type           Level    Vary   Range/Categories                         MAD        Source              
--------------------------------------------------------------------------------------------------------------------------------------
time_delta                     continuous     event    yes    [0.00, 8726400.00]                       2160000.0000 quantile_derived    
amount                         continuous     event    yes    [24.00, 80.00]                           6.7000     quantile_derived    
totalPaymentAmount             continuous     event

In [9]:
preprocessor_artifacts = PreprocessorArtifacts.load(
    path = "../pretrained_models/"
)

In [10]:
model = ProcessLSTM.load(
    path = "../pretrained_models/"
)

In [11]:
model_wrapper = ModelWrapper(
    model=model,
    preprocessor_artifacts=preprocessor_artifacts,
    device=device
)

### --- Process Constraints ---

In [12]:
engine = ProcessModelConstraintEngine.load(
    path = "../pretrained_models/"
)

In [13]:
engine.parallel_sets

[{'Add penalty',
  'Appeal to Judge',
  'Insert Date Appeal to Prefecture',
  'Insert Fine Notification',
  'Payment',
  'Send Fine'},
 {'Add penalty',
  'Appeal to Judge',
  'Insert Date Appeal to Prefecture',
  'Insert Fine Notification',
  'Send Fine'}]

In [14]:
engine.branching_sets

[{'Add penalty',
  'Appeal to Judge',
  'Insert Date Appeal to Prefecture',
  'Insert Fine Notification',
  'Payment',
  'Send Fine',
  'Send for Credit Collection'},
 {'Add penalty',
  'Appeal to Judge',
  'Insert Date Appeal to Prefecture',
  'Insert Fine Notification',
  'Send Fine'},
 {'Appeal to Judge', 'Insert Fine Notification', 'Send Fine'}]

### --- Experiments Generation ---

In [15]:
log_file, original_stdout = set_stdout_to_file(filepath="logs/road_fine-cf_seed777_experiments_ga_ablated_output.txt", console=False)

In [16]:
generator = ExperimentHandler(
    constraint_engine=engine,
)

In [17]:
exp_df_sin, metadata_sin = ExperimentHandler.load("../experiments/cf_generated_experiments_single_desired")
print("Mined using parameters:", metadata_sin["parameters"])

### --- Counterfactuals ---

In [18]:
ga_config = GAConfig(
    w_distance=1.0,
    w_sparsity=1.0,
    w_margin=1.0,
    w_process_violation=0.0,
)
ga_config.validate()

cf_GA = CounterfactualGA(
    ga_config=ga_config,
    feature_config=feature_config,
    model_wrapper=model_wrapper
)

In [19]:
results_sin = generator.run_experiment_df(
    cf_method=cf_GA,
    technique="GA_Ablated_single_desired_seed777",
    exp_df=exp_df_sin,
    df=df,
    feature_config=feature_config,
    case_id_field="case:concept:name",
    sort_field="time:timestamp",
)

Experiments:   0%|          | 0/310 [00:00<?, ?case/s]

In [20]:
results_sin

,template_id,case:concept:name,trace_length,num_desired,num_flexible,DISTANCE,CONT_DISTANCE,CAT_DISTANCE,SPARSITY,PROCESS_VIOLATION,...,best_cf_fitness,best_cf_process_violation,best_cf_distance,best_cf_cat_distance,best_cf_cont_distance,best_cf_sparsity,best_cf_margin_loss,best_cf_margin_loss_na,best_cf_margin_loss_flipped,best_cf_margin_loss_na_flipped
0,0,A19080,5,1,4,0.455766,0.331531,0.58000,0.535417,0.503846,...,0.346955,0.000000,0.138622,0.200,0.077243,0.208333,0.0,0.000000,0.0,0.000000
1,0,S106832,6,1,5,0.468795,0.377591,0.56000,0.547917,0.513333,...,0.420842,0.600000,0.170842,0.200,0.141683,0.250000,0.0,0.950248,0.0,1.000000
2,0,S160509,7,1,4,0.520048,0.385097,0.65500,0.566667,0.708824,...,0.436734,0.764706,0.186734,0.100,0.273468,0.250000,0.0,0.940796,0.0,1.000000
3,1,S71772,3,1,2,0.436613,0.323226,0.55000,0.493750,0.194444,...,0.230692,0.000000,0.105692,0.100,0.111384,0.125000,0.0,0.000000,0.0,0.000000
4,1,V6556,3,1,2,0.425762,0.291525,0.56000,0.450000,0.077778,...,0.183333,0.000000,0.100000,0.200,0.000000,0.083333,0.0,0.000000,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
221,29,S54626,9,1,4,0.261023,0.163296,0.35875,0.324479,0.569048,...,0.169168,0.000000,0.065001,0.025,0.105002,0.104167,0.0,0.000000,0.0,0.000000
222,30,N53905,9,1,4,0.262896,0.168291,0.35750,0.320833,0.600000,...,0.124316,0.619048,0.061816,0.075,0.048632,0.062500,0.0,0.612163,0.0,0.999997
223,30,A373,9,1,4,0.279863,0.148475,0.41125,0.315625,0.578571,...,0.061388,0.523810,0.030138,0.025,0.035277,0.031250,0.0,0.941512,0.0,1.000000
224,30,N56915,9,1,4,0.260602,0.164953,0.35625,0.327604,0.602381,...,0.112318,0.619048,0.028984,0.025,0.032968,0.083333,0.0,0.000000,0.0,0.000000


### --- Cleanup ---

In [21]:
sys.stdout = original_stdout
log_file.close()